In [1]:
from pathlib import Path
import re

import numpy as np
import pandas as pd
from scipy import stats
from statsmodels.tsa.stattools import adfuller

PROJECT_ROOT = Path.cwd()
RAW_DIR = PROJECT_ROOT / "data" / "csv"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)


In [2]:

PRICE_FILES = {
    "gold_futures": {
        "asset": "gold",
        "market": "futures",
        "file": "Gold Futures Historical Data.csv",
        "date_col": "Date",
        "price_col": "Price",
    },
    "silver_futures": {
        "asset": "silver",
        "market": "futures",
        "file": "Silver Futures Historical Data.csv",
        "date_col": "Date",
        "price_col": "Price",
    },
    "platinum_futures": {
        "asset": "platinum",
        "market": "futures",
        "file": "Platinum Futures Historical Data.csv",
        "date_col": "Date",
        "price_col": "Price",
    },
    "palladium_futures": {
        "asset": "palladium",
        "market": "futures",
        "file": "Palladium Futures Historical Data.csv",
        "date_col": "Date",
        "price_col": "Price",
    },
    "gold_spot": {
        "asset": "gold",
        "market": "spot",
        "file": "xauusd_d.csv",
        "date_col": "Date",
        "price_col": "Close",
    },
    "silver_spot": {
        "asset": "silver",
        "market": "spot",
        "file": "xagusd_d.csv",
        "date_col": "Date",
        "price_col": "Close",
    },
    "platinum_spot": {
        "asset": "platinum",
        "market": "spot",
        "file": "xptusd_d.csv",
        "date_col": "Date",
        "price_col": "Close",
    },
    "palladium_spot": {
        "asset": "palladium",
        "market": "spot",
        "file": "xpdusd_d.csv",
        "date_col": "Date",
        "price_col": "Close",
    },
}


In [3]:
def parse_numeric(value):
    """Parse numbers with commas, percent signs, and K/M/B suffixes."""
    if pd.isna(value):
        return np.nan
    if isinstance(value, (int, float, np.integer, np.floating)):
        return float(value)

    text = str(value).strip().replace(",", "").replace("%", "")
    if text in {"", "-", ".", "nan", "None"}:
        return np.nan

    match = re.fullmatch(r"([-+]?\d*\.?\d+)([KMB])?", text, flags=re.IGNORECASE)
    if match:
        number = float(match.group(1))
        suffix = (match.group(2) or "").upper()
        multiplier = {"K": 1_000, "M": 1_000_000, "B": 1_000_000_000}.get(suffix, 1)
        return number * multiplier

    return float(text)


def parse_dates(series):
    """Parse both MM/DD/YYYY futures dates and ISO spot dates."""
    sample = series.dropna().astype(str).head(1)
    if not sample.empty and "/" in sample.iloc[0]:
        parsed = pd.to_datetime(series, format="%m/%d/%Y", errors="coerce")
    else:
        parsed = pd.to_datetime(series, errors="coerce")
    return parsed


def load_price_series(series_name, spec):
    path = RAW_DIR / spec["file"]
    df = pd.read_csv(path, encoding="utf-8-sig")

    date_col = spec["date_col"]
    price_col = spec["price_col"]

    df["date"] = parse_dates(df[date_col])
    df["price"] = df[price_col].map(parse_numeric)
    df = df.dropna(subset=["date"]).copy()

    duplicate_dates = int(df["date"].duplicated().sum())
    weekend_rows = int((df["date"].dt.weekday >= 5).sum())

    df = df.drop_duplicates(subset="date", keep="last")
    df = df.sort_values("date")

    business_days = pd.bdate_range(df["date"].min(), df["date"].max())
    missing_business_days = len(set(business_days) - set(df["date"]))
    max_gap_days = int(df["date"].diff().dt.days.dropna().max()) if len(df) > 1 else 0

    metadata = {
        "series": series_name,
        "asset": spec["asset"],
        "market": spec["market"],
        "source_file": spec["file"],
        "price_col": price_col,
        "start_date": df["date"].min().date(),
        "end_date": df["date"].max().date(),
        "n_obs": len(df),
        "duplicate_dates": duplicate_dates,
        "weekend_rows_raw": weekend_rows,
        "missing_business_days": missing_business_days,
        "max_gap_days": max_gap_days,
        "missing_price": int(df["price"].isna().sum()),
    }

    series = df.set_index("date")["price"].rename(series_name)
    return series, metadata


In [4]:
price_series = {}
metadata_rows = []

for series_name, spec in PRICE_FILES.items():
    series, metadata = load_price_series(series_name, spec)
    price_series[series_name] = series
    metadata_rows.append(metadata)

price_metadata = pd.DataFrame(metadata_rows)
price_metadata


,series,asset,market,source_file,price_col,start_date,end_date,n_obs,duplicate_dates,weekend_rows_raw,missing_business_days,max_gap_days,missing_price
0,gold_futures,gold,futures,Gold Futures Historical Data.csv,Price,2012-01-03,2026-03-31,3257,0,4,463,6,0
1,silver_futures,silver,futures,Silver Futures Historical Data.csv,Price,2012-01-03,2026-03-31,3164,0,23,575,7,0
2,platinum_futures,platinum,futures,Platinum Futures Historical Data.csv,Price,2012-01-02,2026-03-31,4161,0,465,21,4,0
3,palladium_futures,palladium,futures,Palladium Futures Historical Data.csv,Price,2012-01-02,2026-03-31,3944,0,247,20,4,0
4,gold_spot,gold,spot,xauusd_d.csv,Close,2012-01-03,2026-03-31,3675,0,0,41,4,0
5,silver_spot,silver,spot,xagusd_d.csv,Close,2012-01-03,2026-03-31,3675,0,0,41,4,0
6,platinum_spot,platinum,spot,xptusd_d.csv,Close,2012-01-03,2026-03-31,3675,0,0,41,4,0
7,palladium_spot,palladium,spot,xpdusd_d.csv,Close,2012-01-03,2026-03-31,3675,0,0,41,4,0


In [5]:
price_panel_union = pd.concat(price_series.values(), axis=1).sort_index()

# Follow the paper-style daily-return setup: use complete common dates across all futures and spot series.
price_panel = price_panel_union.dropna(how="any").copy()
price_panel.index.name = "date"

panel_summary = pd.DataFrame(
    {
        "metric": [
            "union_start",
            "union_end",
            "union_rows",
            "complete_start",
            "complete_end",
            "complete_rows",
            "complete_weekday_rows",
            "complete_weekend_rows",
        ],
        "value": [
            price_panel_union.index.min().date(),
            price_panel_union.index.max().date(),
            len(price_panel_union),
            price_panel.index.min().date(),
            price_panel.index.max().date(),
            len(price_panel),
            int((price_panel.index.weekday < 5).sum()),
            int((price_panel.index.weekday >= 5).sum()),
        ],
    }
)

price_panel.head(), price_panel.tail(), panel_summary


C:\Users\Administrator\AppData\Local\Temp\ipykernel_28492\3726019888.py:1: Pandas4Warning: Sorting by default when concatenating all DatetimeIndex is deprecated.  In the future, pandas will respect the default of `sort=False`. Specify `sort=True` or `sort=False` to silence this message. If you see this warnings when not directly calling concat, report a bug to pandas.
  price_panel_union = pd.concat(price_series.values(), axis=1).sort_index()


(            gold_futures  silver_futures  platinum_futures  palladium_futures  \
 date                                                                            
 2012-01-03        1600.5          29.572           1436.20              664.8   
 2012-01-04        1612.7          29.097           1426.05              649.9   
 2012-01-05        1620.1          29.296           1421.25              640.4   
 2012-01-06        1616.8          28.683           1406.55              616.1   
 2012-01-09        1608.1          28.782           1426.15              620.0   
 
             gold_spot  silver_spot  platinum_spot  palladium_spot  
 date                                                               
 2012-01-03    1603.88        29.71         1428.5          660.25  
 2012-01-04    1611.73        29.16         1415.2          649.00  
 2012-01-05    1621.43        29.34         1411.0          637.75  
 2012-01-06    1617.55        28.75         1401.5          612.75  
 2012-01-0

In [6]:
# Daily log returns, scaled by 100 as in the reference paper.
return_panel = 100 * np.log(price_panel).diff()
return_panel = return_panel.dropna(how="any")
return_panel.index.name = "date"

return_panel.head(), return_panel.tail()


(            gold_futures  silver_futures  platinum_futures  palladium_futures  \
 date                                                                            
 2012-01-04      0.759371       -1.619289         -0.709235          -2.266774   
 2012-01-05      0.457808        0.681591         -0.337162          -1.472552   
 2012-01-06     -0.203899       -2.114638         -1.039687          -3.868369   
 2012-01-09     -0.539553        0.344558          1.383861           0.631019   
 2012-01-10      1.444648        3.526143          2.830916           2.689318   
 
             gold_spot  silver_spot  platinum_spot  palladium_spot  
 date                                                               
 2012-01-04   0.488244    -1.868578      -0.935408       -1.718583  
 2012-01-05   0.600034     0.615387      -0.297219       -1.748636  
 2012-01-06  -0.239582    -2.031401      -0.675558       -3.998933  
 2012-01-09  -0.413205     0.934748       1.733023        0.488401  
 2012-01-1

In [7]:
pairwise_return_corr = pd.DataFrame(
    {
        asset: [
            return_panel[f"{asset}_futures"].corr(return_panel[f"{asset}_spot"]),
            return_panel[[f"{asset}_futures", f"{asset}_spot"]].dropna().shape[0],
        ]
        for asset in ["gold", "silver", "platinum", "palladium"]
    },
    index=["futures_spot_return_corr", "pair_overlap_rows"],
).T

returns_by_year = return_panel.groupby(return_panel.index.year).size().rename("n_return_obs")

pairwise_return_corr, returns_by_year


(           futures_spot_return_corr  pair_overlap_rows
 gold                       0.854129             3100.0
 silver                     0.868088             3100.0
 platinum                   0.926570             3100.0
 palladium                  0.944448             3100.0,
 date
 2012    251
 2013    252
 2014    252
 2015    252
 2016    252
 2017    197
 2018    198
 2019    198
 2020    201
 2021    200
 2022    199
 2023    198
 2024    198
 2025    198
 2026     54
 Name: n_return_obs, dtype: int64)

In [8]:
def safe_adf(series):
    clean = series.dropna()
    try:
        result = adfuller(clean, autolag="AIC")
        return pd.Series(
            {
                "adf_stat": result[0],
                "adf_pvalue": result[1],
                "adf_lags": result[2],
                "adf_nobs": result[3],
            }
        )
    except Exception:
        return pd.Series({"adf_stat": np.nan, "adf_pvalue": np.nan, "adf_lags": np.nan, "adf_nobs": np.nan})


def describe_return_series(series):
    clean = series.dropna()
    jb_stat, jb_pvalue = stats.jarque_bera(clean)
    adf_values = safe_adf(clean)
    return pd.Series(
        {
            "n_obs": clean.shape[0],
            "max": clean.max(),
            "min": clean.min(),
            "mean": clean.mean(),
            "std": clean.std(ddof=1),
            "skew": stats.skew(clean, bias=False),
            "kurtosis": stats.kurtosis(clean, fisher=False, bias=False),
            "jarque_bera": jb_stat,
            "jb_pvalue": jb_pvalue,
            **adf_values.to_dict(),
        }
    )


return_descriptive_stats = return_panel.apply(describe_return_series).T
return_descriptive_stats


,n_obs,max,min,mean,std,skew,kurtosis,jarque_bera,jb_pvalue,adf_stat,adf_pvalue,adf_lags,adf_nobs
gold_futures,3100.0,5.892647,-12.088085,0.034603,1.127806,-0.754751,12.188195,11158.945074,0.0,-57.177239,0.000000e+00,0.0,3099.0
silver_futures,3100.0,16.877064,-37.646109,0.029986,2.222143,-1.738582,36.855828,149119.360839,0.0,-14.557684,4.851195e-27,17.0,3082.0
platinum_futures,3100.0,11.111510,-21.035473,0.010198,1.848349,-0.662929,13.197937,13611.539954,0.0,-24.531946,0.000000e+00,5.0,3094.0
palladium_futures,3100.0,18.627023,-22.917149,0.025980,2.445828,-0.686725,12.592569,12085.918194,0.0,-25.075940,0.000000e+00,5.0,3094.0
gold_spot,3100.0,5.953571,-10.392029,0.034457,1.084668,-0.600230,10.181931,6823.347228,0.0,-40.696392,0.000000e+00,1.0,3098.0
silver_spot,3100.0,15.226782,-32.894854,0.029926,2.092700,-1.942869,33.988396,125569.572527,0.0,-40.555310,0.000000e+00,1.0,3098.0
platinum_spot,3100.0,10.238735,-20.281132,0.010090,1.736514,-0.800456,13.742875,15184.378952,0.0,-14.442211,7.327639e-27,18.0,3081.0
palladium_spot,3100.0,19.665114,-21.993833,0.026053,2.305547,-0.768500,13.191806,13673.450601,0.0,-23.215450,0.000000e+00,6.0,3093.0


In [9]:
price_panel.to_csv(PROCESSED_DIR / "precious_metals_price_panel_complete.csv", encoding="utf-8-sig")
return_panel.to_csv(PROCESSED_DIR / "precious_metals_daily_log_returns.csv", encoding="utf-8-sig")
price_metadata.to_csv(PROCESSED_DIR / "precious_metals_price_metadata.csv", index=False, encoding="utf-8-sig")
panel_summary.to_csv(PROCESSED_DIR / "precious_metals_price_panel_summary.csv", index=False, encoding="utf-8-sig")
pairwise_return_corr.to_csv(PROCESSED_DIR / "precious_metals_futures_spot_return_correlations.csv", encoding="utf-8-sig")
returns_by_year.to_csv(PROCESSED_DIR / "precious_metals_returns_by_year.csv", encoding="utf-8-sig")
return_descriptive_stats.to_csv(PROCESSED_DIR / "precious_metals_return_descriptive_stats.csv", encoding="utf-8-sig")

saved_files = sorted(p.name for p in PROCESSED_DIR.glob("precious_metals_*.csv"))
saved_files


['precious_metals_daily_log_returns.csv',
 'precious_metals_futures_spot_return_correlations.csv',
 'precious_metals_price_metadata.csv',
 'precious_metals_price_panel_complete.csv',
 'precious_metals_price_panel_summary.csv',
 'precious_metals_return_descriptive_stats.csv',
 'precious_metals_returns_by_year.csv']